# DB7-019: Why did BRB not improve recognition?

This notebook audits the frozen DB7-018 models and predictions. It performs **zero neural fits and zero BRB fits**. All22 subjects and seed42 are included; the inherited protocol is Exercise B, E1 labels1–17,200ms windows,10ms stride, train repetitions1/3/4/6 and test2/5.

## Four diagnostic questions
1. **Recoverability:** Does any independent W/S/I expert predict the correct gesture when inertial or BRB fails? A label-informed selector is a diagnostic reference, not a deployable model or a ceiling for every fusion strategy.
2. **Reliability discrimination:** Does a higher BRB score identify correct predictions, particularly when experts disagree? Compare raw BRB, calibrated BRB and confidence AUROC, Brier score and calibration bins.
3. **Transfer:** Compare OOF-fit, OOF-calibration and final-test correctness, reliability and indicator distributions. These groups differ in training size and repetitions. Their differences do not prove causality.
4. **Weight tracing:** Export global alpha, calibrated reliability, actual normalized weight, predictions and correctness per test window. Summarize recovered and harmed cases per subject, gesture and repetition.

## Input provenance and limits
Attach the output of `beautifulminnd/db7-018-brb-full-34917705725-1`. The exact archived result SHA-256 is checked before any analysis. Source models, labels and predictions remain unchanged. No new raw-data preprocessing is applied.

OOF1/3/4 trained the meta heads; OOF6 calibrated reliability. Both are development data for the complete gate. Do not interpret them as independent validation. Test labels are used for diagnosis only. Correlated windows do not provide independent statistical replicates. Further changes must be selected using training-side folds before another test evaluation.

## Outputs
Oracle/recovery tables, reliability AUROC/Brier tables, calibration bins, rule activation support, effective-weight case tables, OOF-to-test comparisons, all22 per-window compressed traces, gesture/repetition summaries, figures, a report and a verified completion manifest. S2/S20/S3/S15 can be inspected directly through their trace files. CPU execution avoids spending GPU quota on replaying saved predictions.


In [ ]:
from pathlib import Path
SOURCE=Path('/kaggle/working/db7_019_source')
SOURCE.mkdir(exist_ok=True)


## brb_meta.py
Frozen DB7-018 implementation. It contains the original rule matching and fusion functions. This notebook only calls prediction functions; it does not call fit_meta.

In [ ]:
%%writefile /kaggle/working/db7_019_source/brb_meta.py
"""Training-only reliability fusion for DB7 W/S/I expert logits.

Labels are zero based. fit_meta accepts OOF predictions for repetitions 1/3/4/6
only. Repetition 6 is reserved for reliability calibration, not model fitting,
temperature selection or global fusion weights. The caller must supply shifts
computed against each OOF base model's own training-only signal references.

This is nested development, NOT independent meta cross-validation: OOF base
models may share training recordings. Outer test predictions never enter fit.
"""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit, logsumexp, softmax

VERSION = "db7-brb-meta-v1"
EXPERTS = ("W", "S", "I")
FIT_REPS = (1, 3, 4)
CAL_REP = 6
EPS = 1e-9
HEAD_L2 = 0.01
CAL_L2 = 0.01
ALPHA_L2 = 0.005
MIN_EVENTS = 5
MAXITER = 180
RULE_BITS = np.array([[int(x) for x in f"{r:03b}"] for r in range(8)])


def _inputs(logits, shifts, y=None, repetitions=None):
    z = np.asarray(logits, dtype=np.float64)
    s = np.asarray(shifts, dtype=np.float64)
    if z.ndim != 3 or z.shape[1:] != (3, 17) or len(z) == 0:
        raise ValueError("logits must have nonempty shape [N,3,17]")
    if s.shape != z.shape[:2] or not np.isfinite(z).all() or not np.isfinite(s).all():
        raise ValueError("finite shifts [N,3] and logits required")
    if np.any((s < 0) | (s > 1)):
        raise ValueError("training-reference shifts must be in [0,1]")
    if y is None:
        return z, s
    yy = np.asarray(y)
    rr = np.asarray(repetitions)
    if yy.shape != (len(z),) or rr.shape != yy.shape:
        raise ValueError("y and repetitions must have shape [N]")
    if not np.isin(yy, np.arange(17)).all():
        raise ValueError("y must contain zero-based labels 0..16")
    if not np.isin(rr, (*FIT_REPS, CAL_REP)).all():
        raise ValueError("fit_meta accepts only training repetitions 1,3,4,6; test forbidden")
    if set(rr.tolist()) != {1, 3, 4, 6}:
        raise ValueError("all meta-fit repetitions 1/3/4 and calibration repetition 6 required")
    return z, s, yy.astype(np.int64), rr.astype(np.int64)


def _group_weights(groups):
    """Equal repetition weight; do not treat differing window counts as trials."""
    groups = np.asarray(groups)
    levels, counts = np.unique(groups, return_counts=True)
    return np.array([1.0 / (len(levels) * counts[np.searchsorted(levels, g)]) for g in groups])


def _binary_loss(target, predicted, weights):
    p = np.clip(predicted, EPS, 1 - EPS)
    return float(-np.sum(weights * (target * np.log(p) + (1 - target) * np.log1p(-p))))


def _optimization(result):
    return {"success": bool(result.success), "message": str(result.message),
            "iterations": int(result.nit), "objective": float(result.fun)}


def fit_temperatures(logits, y, groups):
    """Positive scalar temperature per expert, using supplied development rows."""
    weights = _group_weights(groups)
    temperatures, diagnostics = [], []
    for j in range(3):
        z = logits[:, j]
        def objective(theta):
            zz = z / np.exp(theta[0])
            p = softmax(zz, axis=1)
            loss = np.sum(weights * (logsumexp(zz, axis=1) - zz[np.arange(len(y)), y]))
            grad = np.sum(weights * (zz[np.arange(len(y)), y] - np.sum(p * zz, axis=1)))
            return float(loss + 0.001 * theta[0] ** 2), np.array([grad + .002 * theta[0]])
        opt = minimize(objective, [0.], jac=True, method="L-BFGS-B",
                       bounds=[(np.log(.05), np.log(20.))], options={"maxiter": MAXITER})
        if not np.isfinite(opt.fun):
            raise RuntimeError("nonfinite temperature optimization")
        temperatures.append(float(np.exp(opt.x[0])))
        diagnostics.append(_optimization(opt))
    return temperatures, diagnostics


def _probabilities(logits, temperatures):
    return softmax(logits / np.asarray(temperatures)[None, :, None], axis=2)


def indicators(probabilities, shifts):
    """Expert entropy, mean pairwise TV and precomputed training deviation."""
    p = np.asarray(probabilities, dtype=float)
    entropy = -np.sum(p * np.log(np.clip(p, EPS, 1)), axis=2) / np.log(17.)
    disagreement = np.zeros(p.shape[:2])
    for j in range(3):
        disagreement[:, j] = sum(.5 * np.abs(p[:, j] - p[:, k]).sum(1)
                                for k in range(3) if k != j) / 2
    return np.clip(np.stack([entropy, disagreement, shifts], axis=-1), 0, 1)


def rule_activations(q):
    """Product reference matching: [...,3] -> [...,8], sum exactly one."""
    q = np.asarray(q, dtype=float)
    if q.shape[-1] != 3 or not np.isfinite(q).all() or np.any((q < 0) | (q > 1)):
        raise ValueError("rule indicators must be finite [...,3] in [0,1]")
    a = np.prod(np.where(RULE_BITS, q[..., None, :], 1 - q[..., None, :]), axis=-1)
    return a / a.sum(axis=-1, keepdims=True)


def rimer_correct(activations, correct_beliefs, return_jacobian=False):
    """Analytical ER/RIMER for complete binary rule conclusions.

    A_n=prod(1-w+w*beta_n), B=prod(1-w), beta_n=(A_n-B)/(A0+A1-2B).
    Normalized activation is rule evidence weight. Returned jacobian is with
    respect to each rule's correctness belief (NOT its logit).
    """
    w = np.asarray(activations, dtype=float)
    b = np.asarray(correct_beliefs, dtype=float)
    if w.shape[-1] != 8 or b.shape != (8,):
        raise ValueError("eight activations and eight correctness beliefs required")
    if not np.isfinite(w).all() or not np.isfinite(b).all() or np.any((b < 0) | (b > 1)):
        raise ValueError("invalid ER beliefs")
    if np.any((w < 0) | (w > 1)) or not np.allclose(w.sum(-1), 1):
        raise ValueError("ER activations must be normalized")
    f1 = 1 - w + w * b
    f0 = 1 - w + w * (1 - b)
    a1, a0, bb = f1.prod(-1), f0.prod(-1), (1 - w).prod(-1)
    denom = a1 + a0 - 2 * bb
    if np.any(denom <= 0):
        raise FloatingPointError("degenerate ER normalization")
    p = np.clip((a1 - bb) / denom, 0, 1)
    if not return_jacobian:
        return p
    # Product excluding each factor handles exact zero factors at rule vertices.
    da1 = np.stack([w[..., k] * np.delete(f1, k, axis=-1).prod(-1) for k in range(8)], -1)
    da0 = -np.stack([w[..., k] * np.delete(f0, k, axis=-1).prod(-1) for k in range(8)], -1)
    jac = (da1 * denom[..., None] - (a1 - bb)[..., None] * (da1 + da0)) / denom[..., None] ** 2
    return p, jac


def _fit_alpha(p, y, weights):
    true_p = p[np.arange(len(p))[:, None], np.arange(3)[None, :], y[:, None]]
    def objective(theta):
        alpha = softmax(theta)
        mixture = np.clip(true_p @ alpha, EPS, 1)
        loss = -np.sum(weights * np.log(mixture)) + ALPHA_L2 * np.sum(theta ** 2)
        da = -np.sum(weights[:, None] * true_p / mixture[:, None], axis=0)
        grad = alpha * (da - alpha @ da) + 2 * ALPHA_L2 * theta
        return float(loss), grad
    opt = minimize(objective, np.zeros(3), jac=True, method="L-BFGS-B",
                   bounds=[(-6, 6)] * 3, options={"maxiter": MAXITER})
    return softmax(opt.x).tolist(), _optimization(opt)


def _raw_head(head, q):
    if head["kind"] == "constant":
        return np.full(len(q), head["value"])
    if head["kind"] == "logistic":
        return expit(np.column_stack([np.ones(len(q)), q - .5]) @ np.asarray(head["parameters"]))
    qq = q.copy()
    if head["kind"] == "brb_no_shift":
        qq[:, 2] = .5
    a = rule_activations(qq)
    beliefs = expit(head["parameters"])
    return a @ beliefs if head["kind"] == "sugeno" else rimer_correct(a, beliefs)


def _fit_head(kind, q, target, weights):
    prior = float((np.sum(target) + .5) / (len(target) + 1))
    if min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS:
        return {"kind": "constant", "requested_kind": kind, "value": prior,
                "fallback": "fewer than five correct or incorrect examples"}
    if kind == "logistic":
        x = np.column_stack([np.ones(len(q)), q - .5])
        center = np.array([logit(prior), 0, 0, 0])
        def objective(theta):
            raw = expit(x @ theta)
            loss = _binary_loss(target, raw, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad = x.T @ (weights * (raw - target)) + 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    else:
        qq = q.copy()
        if kind == "brb_no_shift":
            qq[:, 2] = .5
        a = rule_activations(qq)
        center = np.full(8, logit(prior))
        def objective(theta):
            beliefs = expit(theta)
            if kind == "sugeno":
                raw, jac = a @ beliefs, a
            else:
                raw, jac = rimer_correct(a, beliefs, return_jacobian=True)
            pp = np.clip(raw, EPS, 1 - EPS)
            derivative = weights * (pp - target) / (pp * (1 - pp))
            grad = (derivative @ jac) * beliefs * (1 - beliefs)
            loss = _binary_loss(target, pp, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad += 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    opt = minimize(objective, center, jac=True, method="L-BFGS-B", bounds=[(-10, 10)] * len(center),
                   options={"maxiter": MAXITER, "ftol": 1e-9})
    if not np.isfinite(opt.fun) or not np.isfinite(opt.x).all():
        raise RuntimeError(f"nonfinite {kind} fit")
    return {"kind": kind, "parameters": opt.x.tolist(), "optimization": _optimization(opt), "prior": prior}


def _fit_calibration(raw, target):
    """Monotone logit-affine reliability calibration on repetition 6 only."""
    if len(target) < 20:
        return {"slope": 1., "intercept": 0., "fallback": "fewer than 20 calibration rows"}
    x = logit(np.clip(raw, 1e-5, 1 - 1e-5))
    sparse = min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS
    def objective(theta):
        slope, intercept = theta
        p = expit(slope * x + intercept)
        weights = np.full(len(target), 1 / len(target))
        loss = _binary_loss(target, p, weights) + CAL_L2 * ((slope - 1) ** 2 + intercept ** 2)
        residual = p - target
        grad = np.array([np.mean(residual * x) + 2 * CAL_L2 * (slope - 1),
                         np.mean(residual) + 2 * CAL_L2 * intercept])
        return loss, grad
    opt = minimize(objective, [1., 0.], jac=True, method="L-BFGS-B",
                   bounds=[(1., 1.) if sparse else (0., 5.), (-8., 8.)], options={"maxiter": MAXITER})
    return {"slope": float(opt.x[0]), "intercept": float(opt.x[1]), "optimization": _optimization(opt),
            "fallback": "intercept-only: fewer than five events in one outcome" if sparse else None}


def _calibrated(raw, calibration):
    return expit(calibration["slope"] * logit(np.clip(raw, 1e-5, 1 - 1e-5)) + calibration["intercept"])


def _weights(alpha, reliability):
    unnormalized = np.asarray(alpha)[None, :] * np.clip(reliability, 0, 1)
    denominator = unnormalized.sum(1, keepdims=True)
    return np.divide(unnormalized, denominator, out=np.broadcast_to(alpha, unnormalized.shape).copy(), where=denominator > EPS)


def predict_diagnostics(model, logits, shifts):
    z, s = _inputs(logits, shifts)
    if model.get("version") != VERSION:
        raise ValueError("unsupported meta model version")
    p = _probabilities(z, model["temperatures"])
    q = indicators(p, s)
    reliabilities = {"confidence_weight": p.max(2)}
    raw_reliabilities = {}
    for name, heads in model["heads"].items():
        raw = np.column_stack([_raw_head(heads[j], q[:, j]) for j in range(3)])
        raw_reliabilities[name] = raw
        reliabilities[name] = np.column_stack([_calibrated(raw[:, j], model["reliability_calibrations"][name][j]) for j in range(3)])
    weights = {name: _weights(model["alpha"], r) for name, r in reliabilities.items()}
    return {"expert_probabilities": p, "indicators": q, "raw_reliabilities": raw_reliabilities,
            "reliabilities": reliabilities, "weights": weights, "rule_activations": rule_activations(q)}


def predict_meta(model, logits, shifts):
    d = predict_diagnostics(model, logits, shifts)
    p = d["expert_probabilities"]
    result = {f"expert_{name.lower()}": p[:, j] for j, name in enumerate(EXPERTS)}
    result["mean"] = p.mean(1)
    result["global_weight"] = np.sum(p * np.asarray(model["alpha"])[None, :, None], axis=1)
    result.update({name: np.sum(p * w[:, :, None], axis=1) for name, w in d["weights"].items()})
    return result


def _metrics(y, p):
    confidence, predicted = p.max(1), p.argmax(1)
    correct = predicted == y
    onehot = np.eye(p.shape[1])[y]
    ece = 0.
    for low in np.arange(0, 1, .1):
        mask = (confidence >= low) & (confidence < low + .1 if low < .9 else confidence <= 1)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return {"rows": int(len(y)), "accuracy": float(correct.mean()),
            "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1)).mean()),
            "brier": float(np.sum((p - onehot) ** 2, axis=1).mean()), "ece10": float(ece)}


def _write_csv(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def fit_meta(logits, y, repetitions, shifts, output: Path, metadata=None):
    """Fit and save a JSON-serializable meta model; never supply test rows.

    1/3/4: final temperatures, alpha, reliability heads. Head training inputs use
    leave-one-repetition crossfitted temperatures. 6: monotone scalar reliability
    calibration only. Final temperature is frozen before looking at rep 6.
    """
    z, s, yy, rr = _inputs(logits, shifts, y, repetitions)
    fit = np.isin(rr, FIT_REPS)
    cal = rr == CAL_REP
    temperatures, tdiag = fit_temperatures(z[fit], yy[fit], rr[fit])
    p_final = _probabilities(z, temperatures)
    p_crossfit = np.zeros_like(z[fit])
    crossfits = []
    for held in FIT_REPS:
        inner_train = fit & (rr != held)
        temp, opt = fit_temperatures(z[inner_train], yy[inner_train], rr[inner_train])
        p_crossfit[rr[fit] == held] = _probabilities(z[fit & (rr == held)], temp)
        crossfits.append({"held_repetition": held, "fit_repetitions": sorted(set(rr[inner_train].tolist())),
                          "temperatures": temp, "optimization": opt})
    q_fit = indicators(p_crossfit, s[fit])
    q_cal = indicators(p_final[cal], s[cal])
    # Positive temperature cannot change an expert's class argmax.
    correctness_fit = (z[fit].argmax(2) == yy[fit, None]).astype(float)
    correctness_cal = (z[cal].argmax(2) == yy[cal, None]).astype(float)
    weights_fit = _group_weights(rr[fit])
    alpha, adiag = _fit_alpha(p_crossfit, yy[fit], weights_fit)
    model = {"version": VERSION, "expert_order": list(EXPERTS), "num_classes": 17,
             "temperatures": temperatures, "alpha": alpha, "heads": {}, "reliability_calibrations": {},
             "metadata": metadata or {}, "provenance": {
                 "meta_fit_repetitions": list(FIT_REPS), "reliability_calibration_repetition": CAL_REP,
                 "outer_test_repetitions_forbidden_at_fit": [2, 5], "fit_rows": int(fit.sum()), "calibration_rows": int(cal.sum()),
                 "counts_per_repetition": {str(r): int((rr == r).sum()) for r in sorted(set(rr.tolist()))},
                 "temperatures_crossfit": crossfits, "final_temperature_optimization": tdiag, "alpha_optimization": adiag,
                 "alpha_fit_input": "crossfitted-temperature probabilities from repetitions 1/3/4",
                 "fit_digest": hashlib.sha256(z[fit].tobytes() + yy[fit].tobytes() + s[fit].tobytes() + rr[fit].tobytes()).hexdigest(),
                 "calibration_digest": hashlib.sha256(z[cal].tobytes() + yy[cal].tobytes() + s[cal].tobytes()).hexdigest(),
                 "fixed_hyperparameters": {"head_l2": HEAD_L2, "calibration_l2": CAL_L2, "alpha_l2": ALPHA_L2, "min_events": MIN_EVENTS},
                 "rule_inputs": ["normalized_entropy", "mean_pairwise_total_variation", "training_reference_deviation"],
                 "caveat": "internal development; shared base-model training histories mean this is not independent meta CV"}}
    support_rows, rule_rows, reliability_rows, initial_rows = [], [], [], []
    for name in ("logistic", "sugeno", "brb", "brb_no_shift"):
        model["heads"][name], model["reliability_calibrations"][name] = [], []
        for j, expert in enumerate(EXPERTS):
            head = _fit_head(name, q_fit[:, j], correctness_fit[:, j], weights_fit)
            raw_cal = _raw_head(head, q_cal[:, j])
            calibration = _fit_calibration(raw_cal, correctness_cal[:, j])
            model["heads"][name].append(head)
            model["reliability_calibrations"][name].append(calibration)
            for stage, values in [("before", raw_cal), ("after", _calibrated(raw_cal, calibration))]:
                reliability_rows.append({"method": name, "expert": expert, "stage": stage, "repetition": 6,
                    "scope": "calibration fitting rows; not independent evaluation", "rows": len(raw_cal),
                    "observed_correctness": float(correctness_cal[:, j].mean()), "mean_reliability": float(values.mean()),
                    "brier_binary": float(np.mean((values - correctness_cal[:, j]) ** 2)),
                    "nll_binary": _binary_loss(correctness_cal[:, j], values, np.full(len(values), 1 / len(values)))})
            if name != "logistic":
                qq = q_fit[:, j].copy()
                if name == "brb_no_shift":
                    qq[:, 2] = .5
                a = rule_activations(qq)
                beliefs = np.full(8, head["value"]) if head["kind"] == "constant" else expit(head["parameters"])
                for rule, bits in enumerate(RULE_BITS):
                    initial = float(head.get("prior", head.get("value")))
                    initial_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": 1-initial, "belief_correct": initial,
                        "initial_logit": float(logit(initial)), "rule_weight": 1.0,
                        "antecedent_weights": "1,1,1", "reference_values": "0,1"})
                    rule_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": float(1 - beliefs[rule]), "belief_correct": float(beliefs[rule])})
                    for rep in FIT_REPS:
                        mask = rr[fit] == rep
                        mass = a[mask, rule]
                        support_rows.append({"method": name, "expert": expert, "rule": rule, "repetition": rep,
                            "window_count": int(mask.sum()), "activation_mass": float(mass.sum()),
                            "mean_activation": float(mass.mean()), "windows_activation_above_0_1": int((mass > .1).sum()),
                            "effective_windows_kish_correlated_not_trials": float(mass.sum() ** 2 / max(np.sum(mass ** 2), EPS)),
                            "weighted_correctness": float(mass @ correctness_fit[mask, j] / max(mass.sum(), EPS))})
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / "meta_model.json").write_text(json.dumps(model, indent=2, allow_nan=False), encoding="utf-8")
    (output / "meta_provenance.json").write_text(json.dumps(model["provenance"], indent=2, allow_nan=False), encoding="utf-8")
    _write_csv(output / "meta_rule_initial.csv", initial_rows)
    _write_csv(output / "meta_rule_conclusions.csv", rule_rows)
    _write_csv(output / "meta_rule_support_per_repetition.csv", support_rows)
    _write_csv(output / "meta_reliability_calibration.csv", reliability_rows)
    pred = predict_meta(model, z, s)
    metrics = [{"method": name, "repetition": int(rep),
                "scope": "meta-head development fit" if rep in FIT_REPS else "reliability calibration fit",
                **_metrics(yy[rr == rep], pp[rr == rep])}
               for name, pp in pred.items() for rep in sorted(set(rr.tolist()))]
    _write_csv(output / "meta_development_metrics.csv", metrics)
    return model


## recovery_audit.py
The audit below verifies provenance, replays frozen predictions and measures recoverability, reliability and weight behavior. Missing AUROCs remain missing instead of being replaced by zero.

In [ ]:
%%writefile /kaggle/working/db7_019_source/recovery_audit.py
"""DB7-019: frozen-model diagnostics, never fit or select using test labels."""
from pathlib import Path
import json,hashlib,zipfile
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import brb_meta
SOURCE_SHA='65cf7bf7b914e0eb2f4178e96cffc74d27b6d42dacd694a6eaac42c4aec54eb0'

def auc(y,s):
    y=np.asarray(y,bool);n=y.sum();m=(~y).sum()
    return float((rankdata(s)[y].sum()-n*(n+1)/2)/(n*m)) if n and m else float('nan')

def binary_metrics(y,s):
    y=np.asarray(y,bool);s=np.asarray(s,float)
    return dict(n=len(y),accuracy=float(y.mean()) if len(y) else np.nan,mean_reliability=float(s.mean()) if len(s) else np.nan,
                brier=float(np.mean((s-y)**2)) if len(s) else np.nan,correctness_auc=auc(y,s),
                correct_mean=float(s[y].mean()) if y.any() else np.nan,wrong_mean=float(s[~y].mean()) if (~y).any() else np.nan)

def audit_subject(subject,bundle,model,metadata,out):
    """Return diagnostic rows and export a trace with one row per test window."""
    summary=[];reliability=[];bins=[];support=[];weight_rows=[];indicator_rows=[]
    for scope,mask in [('oof_fit',np.isin(bundle['oof_repetitions'],[1,3,4])),('oof_calibration',bundle['oof_repetitions']==6),('test',None)]:
        prefix='test' if scope=='test' else 'oof'
        z=bundle[prefix+'_logits'];y=bundle[prefix+'_y'];shift=bundle[prefix+'_shifts']
        if mask is not None:z,y,shift=z[mask],y[mask],shift[mask]
        d=brb_meta.predict_diagnostics(model,z,shift);p=d['expert_probabilities'];pred=p.argmax(2);correct=pred==y[:,None]
        rr=d['reliabilities']['brb'];raw=d['raw_reliabilities']['brb'];w=d['weights']['brb'];alpha=np.asarray(model['alpha'])
        fused=(p*w[:,:,None]).sum(1);global_p=(p*alpha[None,:,None]).sum(1)
        bc=fused.argmax(1)==y;gc=global_p.argmax(1)==y;any_correct=correct.any(1)
        disagree=(pred!=pred[:,[0]]).any(1);oracle_opportunity=~correct[:,2]&correct[:,:2].any(1)
        selected=w.argmax(1);selected_correct=correct[np.arange(len(y)),selected]
        summary.append(dict(subject=subject,scope=scope,n=len(y),brb_accuracy=bc.mean(),global_accuracy=gc.mean(),
            expert_w_accuracy=correct[:,0].mean(),expert_s_accuracy=correct[:,1].mean(),expert_i_accuracy=correct[:,2].mean(),
            hard_selector_oracle=any_correct.mean(),all_experts_wrong=(~any_correct).sum(),disagreement_windows=disagree.sum(),
            inertial_error_recoverable=oracle_opportunity.sum(),brb_recovers_inertial_error=(bc&oracle_opportunity).sum(),
            brb_harms_inertial_correct=(~bc&correct[:,2]).sum(),
            best_weight_branch_correct_given_available=selected_correct[any_correct].mean() if any_correct.any() else np.nan,
            recovered_vs_global=(bc&~gc).sum(),harmed_vs_global=(~bc&gc).sum(),
            brb_correct_when_no_expert_top1_correct=(bc&~any_correct).sum()))
        for j,branch in enumerate(['W','S','I']):
            for subset,sel in [('all',np.ones(len(y),bool)),('disagreement',disagree)]:
                for estimator,scores in [('confidence',p[:,j].max(1)),('raw_brb',raw[:,j]),('calibrated_brb',rr[:,j])]:
                    reliability.append(dict(subject=subject,scope=scope,branch=branch,subset=subset,estimator=estimator,**binary_metrics(correct[sel,j],scores[sel])))
                    for k in range(10):
                        hit=sel&(scores>=k/10)&(scores<((k+1)/10) if k<9 else scores<=1)
                        bins.append(dict(subject=subject,scope=scope,branch=branch,subset=subset,estimator=estimator,bin=k,n=int(hit.sum()),mean_score=float(scores[hit].mean()) if hit.any() else np.nan,empirical_correct=float(correct[hit,j].mean()) if hit.any() else np.nan))
            for k,name in enumerate(['uncertainty','disagreement','deviation']):
                scores=d['indicators'][:,j,k]
                indicator_rows.append(dict(subject=subject,scope=scope,branch=branch,indicator=name,mean=scores.mean(),p10=np.quantile(scores,.1),p90=np.quantile(scores,.9),correctness_auc_using_negative_indicator=auc(correct[:,j],-scores)))
            for k in range(8):
                a=d['rule_activations'][:,j,k]
                support.append(dict(subject=subject,scope=scope,branch=branch,rule=k,activation_mass=a.sum(),mean_activation=a.mean(),weighted_correctness=np.sum(a*correct[:,j])/a.sum() if a.sum() else np.nan))
            for case,sel in [('all',np.ones(len(y),bool)),('branch_correct',correct[:,j]),('branch_wrong',~correct[:,j]),('inertial_wrong_other_correct',oracle_opportunity),('inertial_correct_others_wrong',correct[:,2]&~correct[:,:2].any(1))]:
                weight_rows.append(dict(subject=subject,scope=scope,branch=branch,case=case,n=sel.sum(),global_alpha=alpha[j],mean_effective_weight=w[sel,j].mean() if sel.any() else np.nan,mean_reliability=rr[sel,j].mean() if sel.any() else np.nan))
        if scope=='test':
            assert len(metadata)==len(y) and np.array_equal(metadata.gesture.to_numpy()-1,y)
            frame=metadata.copy();frame['brb_pred']=fused.argmax(1)+1;frame['global_pred']=global_p.argmax(1)+1;frame['brb_correct']=bc;frame['global_correct']=gc;frame['any_expert_correct']=any_correct
            frame['recovered_vs_global']=bc&~gc;frame['harmed_vs_global']=~bc&gc
            for j,b in enumerate(['W','S','I']):
                frame[b+'_pred']=pred[:,j]+1;frame[b+'_correct']=correct[:,j];frame[b+'_alpha']=alpha[j];frame[b+'_reliability']=rr[:,j];frame[b+'_weight']=w[:,j]
                for k,name in enumerate(['uncertainty','disagreement','deviation']):frame[b+'_'+name]=d['indicators'][:,j,k]
            frame.to_csv(out/f'S{subject:02d}_window_trace.csv.gz',index=False,compression='gzip')
            frame.groupby(['gesture','native_repetition']).agg(windows=('brb_correct','size'),brb_correct=('brb_correct','sum'),recovered=('recovered_vs_global','sum'),harmed=('harmed_vs_global','sum')).reset_index().to_csv(out/f'S{subject:02d}_gesture_repetition.csv',index=False)
    return summary,reliability,bins,support,weight_rows,indicator_rows

def main(input_root='/kaggle/input',output='/kaggle/working/db7_019_recovery_audit'):
    out=Path(output);out.mkdir(parents=True,exist_ok=True)
    archives=list(Path(input_root).rglob('db7_brb_full_20260915T013327619764Z.zip'))
    if len(archives)!=1:raise RuntimeError(f'Expected exactly one DB7-018 source archive, found {len(archives)}. Attach the specified source notebook output.')
    source=archives[0];h=hashlib.sha256()
    with source.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    assert h.hexdigest()==SOURCE_SHA,'Source archive differs from verified DB7-018'
    tables=[[] for _ in range(6)];subjects=[]
    with zipfile.ZipFile(source) as z:
        completion=json.loads(z.read('completion.json'));assert completion['success'] and completion['neural_fits']==374 and completion['meta_completed']==22
        names=[n for n in z.namelist() if n.startswith('full/') and n.endswith('/subject_bundle.npz')]
        assert len(names)==22
        for name in sorted(names):
            base=name.rsplit('/',1)[0];model=json.loads(z.read(base+'/meta/meta_model.json'));subject=int(model['metadata']['subject']);subjects.append(subject)
            import io
            with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as npz:bundle={k:npz[k] for k in npz.files}
            metadata=pd.read_csv(io.BytesIO(z.read(base+'/test_metadata.csv')))
            rows=audit_subject(subject,bundle,model,metadata,out)
            original=pd.read_csv(io.BytesIO(z.read(base+'/results/method_metrics.csv'))).set_index('method')
            replay=next(row for row in rows[0] if row['scope']=='test')
            assert np.isclose(replay['brb_accuracy'],original.loc['brb','accuracy'],rtol=0,atol=1e-12), 'Frozen replay differs from source'
            for target,rows_i in zip(tables,rows):target.extend(rows_i)
            print('AUDITED',subject,flush=True)
    assert sorted(subjects)==list(range(1,23))
    names=['oracle_recovery','reliability_discrimination','calibration_bins','rule_support','weight_cases','indicator_distributions']
    for name,rows in zip(names,tables):pd.DataFrame(rows).to_csv(out/(name+'.csv'),index=False)
    s=pd.DataFrame(tables[0]);test=s[s.scope=='test'];means=test.select_dtypes('number').drop(columns='subject').mean()
    rel=pd.DataFrame(tables[1]);pivot=rel.query('subset=="all" and estimator=="calibrated_brb"').pivot(index=['subject','branch'],columns='scope',values=['accuracy','mean_reliability','brier','correctness_auc']);pivot.to_csv(out/'oof_to_test_transfer.csv')
    import matplotlib;matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    ax=test.set_index('subject')[['expert_i_accuracy','brb_accuracy','hard_selector_oracle']].mul(100).plot(figsize=(12,5),marker='o');ax.set_ylabel('Window accuracy (%)');ax.set_title('Frozen DB7-018 predictions: oracle is label-informed, not deployable');ax.figure.tight_layout();ax.figure.savefig(out/'oracle_gap.png',dpi=160);plt.close(ax.figure)
    ax=rel.query('scope=="test" and subset=="disagreement"').groupby(['branch','estimator']).correctness_auc.mean().unstack().plot.bar(figsize=(9,5),ylim=(0,1));ax.set_ylabel('Mean subject correctness AUROC');ax.figure.tight_layout();ax.figure.savefig(out/'reliability_auc.png',dpi=160);plt.close(ax.figure)
    notes='''# DB7-019 frozen BRB recovery audit
No neural training, rule fitting or hyperparameter selection. All22 subjects, seed42, original200ms/10ms split. Test labels are used only for diagnosis.

The hard-selector oracle is an upper bound for choosing one expert top-1, not for arbitrary probability fusion. OOF-fit diagnostics are in-sample for the meta learner; OOF6 was used for reliability calibration. Neither is independent validation of the complete gate. OOF-to-test changes mix training-size and repetition changes and cannot establish causality. Overlapping windows are not independent samples. AUROC is missing when a subset has only one correctness class. Actual expert weights, rather than global alpha alone, are saved for every test window.

Questions: Is there recoverable complementarity? Can reliability rank correct experts during disagreement? Does calibration transfer to final experts? Which weights recover or harm individual windows?

All means below give each subject equal weight. Counts shown as means are not total counts; consult oracle_recovery.csv for exact counts. No improvement is claimed by this diagnostic notebook.
'''
    (out/'REPORT.md').write_text(notes+'\n```\n'+means.to_string()+'\n```\n',encoding='utf-8')
    manifest=dict(experiment_id='DB7-019',success=True,subjects=sorted(subjects),source_archive_sha256=SOURCE_SHA,neural_fits=0,meta_fits=0,test_used_for_fitting=False)
    (out/'completion.json').write_text(json.dumps(manifest,indent=2))
    import shutil;shutil.make_archive(str(out),'zip',out)
    print(json.dumps(manifest),flush=True)
if __name__=='__main__':main()


## Run the frozen audit
Failure to locate or verify the original archive stops execution. No training fallback is allowed.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(SOURCE/'recovery_audit.py')],check=True)


In [ ]:
import pandas as pd
from IPython.display import display,Markdown,Image
OUT=Path('/kaggle/working/db7_019_recovery_audit')
display(Markdown((OUT/'REPORT.md').read_text()))
display(pd.read_csv(OUT/'oracle_recovery.csv').query('scope == "test"'))
display(Image(filename=str(OUT/'oracle_gap.png')))
display(Image(filename=str(OUT/'reliability_auc.png')))
